In [134]:
import os
from IPython.display import clear_output
import time
import pandas as pd
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable

In [311]:
# dylan instance details 
NEO4J_URI='neo4j+s://5fe6b298.databases.neo4j.io'
NEO4J_USERNAME='neo4j'
NEO4J_PASSWORD='V2S2unBr-ex60KUDHFzMH_KLkiPggGUR9UkEZYreAPM'
AURA_INSTANCEID='5fe6b298'
AURA_INSTANCENAME='Instance01'


driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

In [361]:
fixtures_df['match_id'] = fixtures_df['GW']+"_"+fixtures_df['Home']+"_vs_"+fixtures_df['Away']


In [368]:
with driver.session(database="neo4j") as session:
    result = session.run("MATCH (n:Match) RETURN n.id AS match_id")
    matches = [record["match_id"] for record in result]
neo_matches = set(matches)
fixtures_matches = set(fixtures_df['match_id'])

missing = fixtures_matches-neo_matches

nw_fixtures_df = fixtures_df[fixtures_df['match_id'].isin(missing)]

In [369]:
nw_fixtures_df

,GW,Day,Date,Time,Home,xG,Score,xG.1,Away,Attendance,Venue,Referee,Match Report,Notes,Home_Score,Away_Score,match_id
380,GW35,Sat,2024-04-27,20:00,Aston Villa,1.1,2–2,1.6,Chelsea,42354.0,Villa Park,Craig Pawson,Match Report,NaN,2,2,GW35_Aston Villa_vs_Chelsea
381,GW35,Sun,2024-04-28,14:00,Tottenham,2.3,2–3,1.0,Arsenal,61554.0,Tottenham Hotspur Stadium,Michael Oliver,Match Report,NaN,2,3,GW35_Tottenham_vs_Arsenal
382,GW35,Sun,2024-04-28,14:00,Bournemouth,1.9,3–0,0.9,Brighton,11188.0,Vitality Stadium,Paul Tierney,Match Report,NaN,3,0,GW35_Bournemouth_vs_Brighton
383,GW35,Sun,2024-04-28,16:30,Nott'ham Forest,1.9,0–2,0.9,Manchester City,29677.0,The City Ground,Simon Hooper,Match Report,NaN,0,2,GW35_Nott'ham Forest_vs_Manchester City
385,GW26,Thu,2024-05-02,19:30,Chelsea,2.2,2–0,1.6,Tottenham,39086.0,Stamford Bridge,Robert Jones,Match Report,NaN,2,0,GW26_Chelsea_vs_Tottenham
387,GW36,Fri,2024-05-03,20:00,Luton Town,0.8,1–1,1.4,Everton,11873.0,Kenilworth Road Stadium,Tim Robinson,Match Report,NaN,1,1,GW36_Luton Town_vs_Everton
388,GW36,Sat,2024-05-04,12:30,Arsenal,3.3,3–0,0.5,Bournemouth,60348.0,Emirates Stadium,David Coote,Match Report,NaN,3,0,GW36_Arsenal_vs_Bournemouth
389,GW36,Sat,2024-05-04,15:00,Burnley,1.0,1–4,3.4,Newcastle Utd,21781.0,Turf Moor,Anthony Taylor,Match Report,NaN,1,4,GW36_Burnley_vs_Newcastle Utd
390,GW36,Sat,2024-05-04,15:00,Sheffield Utd,2.2,1–3,1.4,Nott'ham Forest,28516.0,Bramall Lane,Chris Kavanagh,Match Report,NaN,1,3,GW36_Sheffield Utd_vs_Nott'ham Forest
391,GW36,Sat,2024-05-04,15:00,Brentford,0.6,0–0,1.3,Fulham,17090.0,Gtech Community Stadium,Graham Scott,Match Report,NaN,0,0,GW36_Brentford_vs_Fulham


In [373]:
def listdir(directory:str) -> list:
    return [x for x in os.listdir(directory) if x[0] != '.']

data_directory = os.path.join(os.getcwd(),'data')
for season_directory in listdir(data_directory):
    print(season_directory)
    
    fixtures_df = pd.read_csv(os.path.join(data_directory, season_directory, f'fixtures_{season_directory}.csv'))
    match_reports = list()
    team_dict = dict()

    with open(os.path.join(data_directory, season_directory, f'match_report_{season_directory}.pkl'), "rb") as f:
        match_reports = pickle.load(f)
    # with open(os.path.join(data_directory, season_directory, f'team_dict_{season_directory}.pkl'), "rb") as f:
    #     team_dict = pickle.load(f)
    with open(os.path.join(data_directory, season_directory, f'teams{season_directory}.pkl'), "rb") as f:
        team_dict = pickle.load(f)
    with open(os.path.join(data_directory, season_directory, f'players{season_directory}.pkl'), "rb") as f:
        player_details = pickle.load(f)

    #clean up
    fixtures_df = fixtures_df[fixtures_df['Wk'].notna()].reset_index(drop=True)
    fixtures_df[['Home_Score', 'Away_Score']] = fixtures_df['Score'].str.split('–', expand=True)
    fixtures_df['Wk'] = "GW" + fixtures_df['Wk'].astype(int).astype(str)
    fixtures_df = fixtures_df.rename(columns={'Wk':'GW'})

    print('the length of fixtures df is '+(str(len(fixtures_df))))

    #now split teams_df into list of player names for a team and a list of players with their data
    # team_players = dict()
    # player_details = list()

    # basically need to whats commented out below
    for team_name, players_names_list in team_dict.items():
        for player in players_names_list:
            if player in player_details:
                player_details[player]['Team'] = team_name
    for player in player_details.keys():
        player_details[player]['Name'] = player
    player_details = list(player_details.values())
    
    with driver.session(database="neo4j") as session:
        for gw in range(1, 39): 
            query = """
            MERGE (s:Season {name: $season_name})
            CREATE (gw:GameWeek {gw_number: $gw_number, season:$season_name)-[:IS_A_GW]->(s)
            """
            session.run(query, {"season_name": season_directory, "gw_number": f"GW{gw}"})

        
        query = """
                UNWIND $players AS player
                MERGE (p:Player {name: player.Name})
                SET p.position = player.position,
                    p.footed = player.footed,
                    p.height = player.height,
                    p.weight = player.weight,
                    p.dob = player.dob,
                    p.birth_place = player.birth_place,
                    p.national_team = player.national_team,
                    p.club = player.club,
                    p.wages = player.wages,
                    p.contract_expiry = player.contract_expiry,
                    p.img = player.img
    
                MERGE (team:Team {name: player.Team})
                MERGE (squad: Squad{name: player.Team + " " + $season_name})
                MERGE (team)-[:HAS_SQUAD]->(squad)
                MERGE (p)-[:IS_IN_SQUAD]->(squad)
        """
        
        session.run(query, players=player_details, season_name = season_directory)
        for i, row in fixtures_df[:len(match_reports).iterrows():

            match_id = f"{row['GW']}_{row['Home']}_vs_{row['Away']}"
            match_creation_query = """
                CREATE (match:Match {id: $match_id})
                SET match.date = $date,
                    match.time = $time
                MERGE (venue:Venue {name: $venue})
                MERGE (match)-[:PLAYED_AT]->(venue)
                WITH match
                MATCH (home_team:Squad{name: $home_team + " " + $season_name})
                MATCH (away_team:Squad{name: $away_team + " " + $season_name})
                MATCH (game_week:GameWeek{gw_number: $gameweek, season: $season_name})
                MERGE (match)-[:HOME_TEAM]->(home_team)
                MERGE (match)-[:AWAY_TEAM]->(away_team)
                MERGE (match)-[:IS_OF_GW]->(game_week)
                """
            # Data for the match
            data = {
                "match_id": match_id,
                "date": row["Date"],
                "time": row["Time"],
                "home_team": row["Home"],
                "away_team": row["Away"],
                "gameweek": row["GW"],
                "venue": row["Venue"]
            }
            
            if row["Match Report"] == "Match Report":
                data.update({
                    "home_score": row["Home_Score"],
                    "away_score": row["Away_Score"],
                    "referee": row["Referee"]
                })
                match_creation_query += """
                SET match.home_score = $home_score,
                    match.away_score = $away_score
                MERGE (referee:Referee {name: $referee})
                CREATE (match)-[:HAS_REFEREE]->(referee)
                """
                if "xG" in row:
                    data.update({
                        "home_expected": row["xG"],
                        "away_expected": row["xG.1"]
                    })
                    match_creation_query += """
                    SET match.home_expected = $home_expected,
                        match.away_expected =  $away_expected
                    """
            # Create the match and basic relationships
            session.run(match_creation_query,data, season_name = season_directory)
            
            # Create home and away team squad nodes

            match_id = f"{row['GW']}_{row['Home']}_vs_{row['Away']}"
            home_team_stats = match_reports[i][0][0][:-1] 
            away_team_stats = match_reports[i][1][0][:-1] 
            home_team_goaly_stats = match_reports[i][0][1] 
            away_team_goaly_stats = match_reports[i][1][1]
    
            home_team_name = row['Home']
            away_team_name = row['Away']
    
            # Add Home Team Player Stats
            for team_name, team_stats in [(home_team_name,home_team_stats), (away_team_name, away_team_stats)]:
                for _, player_stats in team_stats.iterrows():
                    player_query = """
                    MATCH (player:Player {name: $player_name})
                    MATCH (squad: Squad{name: $team_name + " " + $season_name})
                    MATCH (player)-[:IS_IN_SQUAD]->(squad)
                    MATCH (match:Match {id: $match_id})
                    MERGE (player)-[h:PLAYS_IN]->(match)
                    SET h.goals = $goals,
                        h.position = $position,
                        h.assists = $assists,
                        h.penalties = $penalties,
                        h.shots = $shots,
                        h.shots_on_target = $shots_on_target,
                        h.touches = $touches,
                        h.tackles = $tackles,
                        h.interceptions = $interceptions,
                        h.blocks = $blocks,
                        h.xg = $xg,
                        h.npxg = $npxg,
                        h.xag = $xag,
                        h.sca = $sca,
                        h.gca = $gca,
                        h.passes_completed = $passes_completed,
                        h.passes_attempted = $passes_attempted,
                        h.pass_accuracy = $pass_accuracy,
                        h.progressive_passes = $progressive_passes,
                        h.carries = $carries,
                        h.progressive_carries = $progressive_carries
                    """
                    session.run(player_query, {
                        "player_name": player_stats[('Unnamed: 0_level_0', 'Player')],
                        "position": player_stats[('Unnamed: 3_level_0', 'Pos')],
                        "match_id": match_id,
                        "team_name": team_name,
                        "goals": player_stats[('Performance', 'Gls')],
                        "assists": player_stats[('Performance', 'Ast')],
                        "penalties": player_stats[('Performance', 'PK')],
                        "shots": player_stats[('Performance', 'Sh')],
                        "shots_on_target": player_stats[('Performance', 'SoT')],
                        "touches": player_stats[('Performance', 'Touches')],
                        "tackles": player_stats[('Performance', 'Tkl')],
                        "interceptions": player_stats[('Performance', 'Int')],
                        "blocks": player_stats[('Performance', 'Blocks')],
                        "xg": player_stats[('Expected', 'xG')],
                        "npxg": player_stats[('Expected', 'npxG')],
                        "xag": player_stats[('Expected', 'xAG')],
                        "sca": player_stats[('SCA', 'SCA')],
                        "gca": player_stats[('SCA', 'GCA')],
                        "passes_completed": player_stats[('Passes', 'Cmp')],
                        "passes_attempted": player_stats[('Passes', 'Att')],
                        "pass_accuracy": player_stats[('Passes', 'Cmp%')],
                        "progressive_passes": player_stats[('Passes', 'PrgP')],
                        "carries": player_stats[('Carries', 'Carries')],
                        "progressive_carries": player_stats[('Carries', 'PrgC')],
                        "season_name": season_directory
                    })

            for team_name, team_goaly_stats in [(home_team_name, home_team_goaly_stats),(away_team_name, away_team_goaly_stats)]:
                for _, goaly_stat in team_goaly_stats.iterrows():
                    goalie_query = """
                    MATCH (goalie:Player {name: $goalie_name})
                    MATCH (squad: Squad{name: $team_name + " " + $season_name})
                    MATCH (goalie)-[:IS_IN_SQUAD]->(squad)
                    MATCH (match:Match {id: $match_id})
                    MERGE (goalie)-[h:PLAYS_IN]->(match)
                    SET goalie.minutes_played = $minutes_played,
                        goalie.sota = $sota,
                        goalie.goals_allowed = $goals_allowed,
                        goalie.saves = $saves,
                        goalie.save_percentage = $save_percentage,
                        goalie.psxg = $psxg,
                        goalie.passes_completed = $passes_completed,
                        goalie.passes_attempted = $passes_attempted,
                        goalie.pass_accuracy = $pass_accuracy,
                        goalie.gk_passes_attempted = $gk_passes_attempted,
                        goalie.throws = $throws,
                        goalie.launch_percentage = $launch_percentage,
                        goalie.launch_average_length = $launch_average_length,
                        goalie.goal_kicks_attempted = $goal_kicks_attempted,
                        goalie.goal_kick_launch_percentage = $goal_kick_launch_percentage,
                        goalie.goal_kick_average_length = $goal_kick_average_length,
                        goalie.crosses_opportunities = $crosses_opportunities,
                        goalie.crosses_stops = $crosses_stops,
                        goalie.crosses_stop_percentage = $crosses_stop_percentage,
                        goalie.opa = $opa,
                        goalie.opa_average_distance = $opa_average_distance
                    """
                    session.run(goalie_query, {
                        "goalie_name": goaly_stat[('Unnamed: 0_level_0', 'Player')],
                        "match_id": match_id,
                        "team_name": team_name,
                        "minutes_played": goaly_stat[('Unnamed: 3_level_0', 'Min')],
                        "sota": goaly_stat[('Shot Stopping', 'SoTA')],
                        "goals_allowed": goaly_stat[('Shot Stopping', 'GA')],
                        "saves": goaly_stat[('Shot Stopping', 'Saves')],
                        "save_percentage": goaly_stat[('Shot Stopping', 'Save%')],
                        "psxg": goaly_stat[('Shot Stopping', 'PSxG')],
                        "passes_completed": goaly_stat[('Launched', 'Cmp')],
                        "passes_attempted": goaly_stat[('Launched', 'Att')],
                        "pass_accuracy": goaly_stat[('Launched', 'Cmp%')],
                        "gk_passes_attempted": goaly_stat[('Passes', 'Att (GK)')],
                        "throws": goaly_stat[('Passes', 'Thr')],
                        "launch_percentage": goaly_stat[('Passes', 'Launch%')],
                        "launch_average_length": goaly_stat[('Passes', 'AvgLen')],
                        "goal_kicks_attempted": goaly_stat[('Goal Kicks', 'Att')],
                        "goal_kick_launch_percentage": goaly_stat[('Goal Kicks', 'Launch%')],
                        "goal_kick_average_length": goaly_stat[('Goal Kicks', 'AvgLen')],
                        "crosses_opportunities": goaly_stat[('Crosses', 'Opp')],
                        "crosses_stops": goaly_stat[('Crosses', 'Stp')],
                        "crosses_stop_percentage": goaly_stat[('Crosses', 'Stp%')],
                        "opa": goaly_stat[('Sweeper', '#OPA')],
                        "opa_average_distance": goaly_stat[('Sweeper', 'AvgDist')],
                        "season_name":season_directory
                    })

2021-2022
the length of fixtures df is 380
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266

IndexError: list index out of range

In [347]:
row['Home']

'Bournemouth'

In [276]:
team_dict['Manchester City']

{'Reuell Walters': {'position': 'DF',
  'footed': 'Right',
  'height': '191cm',
  'weight': '74kg',
  'dob': '2002-04-28',
  'birth_place': 'England, United Kingdom',
  'national_team': 'Lithuanialt',
  'club': 'Sheffield United',
  'wages': '￡ 30,000 Weekly',
  'contract_expiry': 'June 2026',
  'img': 'https://fbref.com/req/202302030/images/headshots/e4e0deaa_2022.jpg'},
 'Luis Díaz': {'position': 'FW-MF (AM-WM, left)',
  'footed': 'Right',
  'height': '180cm',
  'weight': '72kg',
  'dob': '1997-01-13',
  'birth_place': 'Barrancas, Colombia',
  'national_team': 'Colombiaco',
  'club': 'Liverpool',
  'wages': '￡ 55,000 Weekly',
  'contract_expiry': 'June 2027',
  'img': 'https://fbref.com/req/202302030/images/headshots/4a1a9578_2022.jpg'},
 'Trent Alexander-Arnold': {'position': 'DF (FB, right)',
  'footed': 'Right',
  'height': '175cm',
  'weight': '68kg',
  'dob': '1998-10-07',
  'birth_place': 'Liverpool, England, United Kingdom',
  'national_team': 'Englandeng',
  'club': 'Liverpoo

In [380]:
with driver.session(database="neo4j") as session:
    for i, row in fixtures_df[:len(match_reports)].iterrows():
        match_id = f"{row['GW']}_{row['Home']}_vs_{row['Away']}"
        match_creation_query = """
            CREATE (match:Match {id: $match_id})
            SET match.date = $date,
                match.time = $time
            MERGE (venue:Venue {name: $venue})
            MERGE (match)-[:PLAYED_AT]->(venue)
            WITH match
            MATCH (home_team:Squad{name: $home_team + " " + $season_name})
            MATCH (away_team:Squad{name: $away_team + " " + $season_name})
            MATCH (game_week:GameWeek{gw_number: $gameweek, season: $season_name})
            MERGE (match)-[:HOME_TEAM]->(home_team)
            MERGE (match)-[:AWAY_TEAM]->(away_team)
            MERGE (match)-[:IS_OF_GW]->(game_week)
            """
        # Data for the match
        data = {
            "match_id": match_id,
            "date": row["Date"],
            "time": row["Time"],
            "home_team": row["Home"],
            "away_team": row["Away"],
            "gameweek": row["GW"],
            "venue": row["Venue"]
        }
        
        if row["Match Report"] == "Match Report":
            data.update({
                "home_score": row["Home_Score"],
                "away_score": row["Away_Score"],
                "referee": row["Referee"]
            })
            match_creation_query += """
            SET match.home_score = $home_score,
                match.away_score = $away_score
            MERGE (referee:Referee {name: $referee})
            CREATE (match)-[:HAS_REFEREE]->(referee)
            """
            if "xG" in row:
                data.update({
                    "home_expected": row["xG"],
                    "away_expected": row["xG.1"]
                })
                match_creation_query += """
                SET match.home_expected = $home_expected,
                    match.away_expected =  $away_expected
                """
        # Create the match and basic relationships
        session.run(match_creation_query,data, season_name = season_directory)
        

In [382]:
# Transform fixtures_df to a list of dictionaries for UNWIND
match_data = []
for i, row in fixtures_df[:len(match_reports)].iterrows():
    match_entry = {
        "match_id": f"{row['GW']}_{row['Home']}_vs_{row['Away']}",
        "date": row["Date"],
        "time": row["Time"],
        "home_team": row["Home"],
        "away_team": row["Away"],
        "gameweek": row["GW"],
        "venue": row["Venue"],
        "has_match_report": row["Match Report"] == "Match Report",
        "home_score": row["Home_Score"] if row["Match Report"] == "Match Report" else None,
        "away_score": row["Away_Score"] if row["Match Report"] == "Match Report" else None,
        "referee": row["Referee"] if row["Match Report"] == "Match Report" else None,
        "home_expected": row["xG"] if "xG" in row and row["Match Report"] == "Match Report" else None,
        "away_expected": row["xG.1"] if "xG.1" in row and row["Match Report"] == "Match Report" else None,
    }
    match_data.append(match_entry)


# UNWIND query
query = """
UNWIND $matches AS match_data
CREATE (match:Match {id: match_data.match_id})
SET match.date = match_data.date,
    match.time = match_data.time
MERGE (venue:Venue {name: match_data.venue})
MERGE (match)-[:PLAYED_AT]->(venue)
WITH match, match_data
MATCH (home_team:Squad {name: match_data.home_team + " " + $season_name})
MATCH (away_team:Squad {name: match_data.away_team + " " + $season_name})
MATCH (game_week:GameWeek {gw_number: match_data.gameweek, season: $season_name})
MERGE (match)-[:HOME_TEAM]->(home_team)
MERGE (match)-[:AWAY_TEAM]->(away_team)
MERGE (match)-[:IS_OF_GW]->(game_week)
FOREACH (_ IN CASE WHEN match_data.has_match_report THEN [1] ELSE [] END |
    SET match.home_score = match_data.home_score,
        match.away_score = match_data.away_score
    FOREACH (_ IN CASE WHEN match_data.referee IS NOT NULL THEN [1] ELSE [] END |
        MERGE (referee:Referee {name: match_data.referee})
        MERGE (match)-[:HAS_REFEREE]->(referee)
    )
    FOREACH (_ IN CASE WHEN match_data.home_expected IS NOT NULL THEN [1] ELSE [] END |
        SET match.home_expected = match_data.home_expected,
            match.away_expected = match_data.away_expected
    )
)
"""

# Run the query with the match data
with driver.session(database="neo4j") as session:
    session.run(query, matches=match_data, season_name=season_directory)
